# NB00 — Upstream Raw Data Pipeline
**Pipeline:** Generalised Metabolite–Metagenomics Correlation Study  
**Purpose:** Acquire raw sequencing reads + mzML metabolomics data, run QC/trimming,  
taxonomic profiling (MetaPhlAn4), functional profiling (HUMAnN3), and mzML feature extraction.  
Outputs drop-in TSV files into `Data/` consumed by NB01 with no structural changes needed.  
**When to run:** Once per new cohort. NB01–NB06 iterate freely on cached TSVs.  
**Estimated runtime:** 4–48 hours depending on cohort size and available CPU cores.  
---

## 0 · Prerequisites Check

In [ ]:
import sys, subprocess, shutil
from pathlib import Path

# Tool availability check — all must be present before running
TOOLS = {
    'fastq-dump':    'SRA toolkit (fasterq-dump preferred)',
    'fastp':         'QC + adapter trimming',
    'metaphlan':     'Taxonomic profiling (MetaPhlAn4)',
    'humann':        'Functional/pathway profiling (HUMAnN3)',
    'multiqc':       'Aggregate QC reports',
    'python':        'Python (this environment)',
}
print('Tool availability check:')
missing = []
for tool, desc in TOOLS.items():
    found = shutil.which(tool)
    status = '✓' if found else '✗ MISSING'
    print(f'  {status}  {tool:<18}  {desc}')
    if not found:
        missing.append(tool)

if missing:
    print(f'\nInstall missing tools before running:')
    print('  conda install -c bioconda fastp samtools')
    print('  pip install metaphlan humann pysradb multiqc')
else:
    print('\nAll tools available. Proceed.')


---
## 1 · Configuration

In [ ]:
from pathlib import Path

# --- Project paths ---
BASE_DIR      = Path(r'C:\Users\andna\Documents\KI')
DATA_DIR      = BASE_DIR / 'Data'
RAW_DIR       = BASE_DIR / 'RawData'          # raw FASTQ files
QC_DIR        = BASE_DIR / 'QC'               # FastP + MultiQC reports
TRIMMED_DIR   = BASE_DIR / 'Trimmed'          # post-QC FASTQs
PROFILE_DIR   = BASE_DIR / 'Profiles'         # MetaPhlAn + HUMAnN outputs
MZML_DIR      = BASE_DIR / 'mzML'             # raw metabolomics files
MTB_OUT_DIR   = BASE_DIR / 'MetabolomicsOut'  # processed metabolite tables

for d in [DATA_DIR, RAW_DIR, QC_DIR, TRIMMED_DIR, PROFILE_DIR, MZML_DIR, MTB_OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- SRA accessions to download (replace with your study accessions) ---
# Format: {dataset_name: [SRR_accession_list]}
SRA_ACCESSIONS = {
    # Example: 'MY_COHORT_2025': ['SRR000001', 'SRR000002'],
    # To get accessions from a BioProject:
    #   pysradb search --db 'SRA' --query 'PRJNAXXXXXX' | grep SRR
}

# --- CPU / memory settings ---
CPUS     = 8
MEM_GB   = 24   # HUMAnN3 recommends >=16 GB RAM

print('Configuration ready.')
print(f'  RAW_DIR    : {RAW_DIR}')
print(f'  DATA_DIR   : {DATA_DIR}')
print(f'  CPUs       : {CPUS}')
print(f'  Cohorts    : {list(SRA_ACCESSIONS.keys())}')


---
## 2 · SRA Download

In [ ]:
# Download FASTQ files from NCBI SRA using fasterq-dump (preferred) or fastq-dump
# Progress bars and parallel downloads via concurrent.futures

import concurrent.futures, subprocess
from tqdm import tqdm

def download_sra(accession, out_dir, cpus=4):
    out_dir.mkdir(parents=True, exist_ok=True)
    fastq_file = out_dir / f'{accession}_1.fastq'
    if fastq_file.exists():
        return accession, True, 'already downloaded'
    cmd = ['fasterq-dump', accession,
           '--outdir', str(out_dir),
           '--threads', str(cpus),
           '--split-files', '--skip-technical']
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
        success = result.returncode == 0
        return accession, success, result.stderr[:200] if not success else 'OK'
    except Exception as e:
        return accession, False, str(e)

all_accessions = [(ds, acc) for ds, accs in SRA_ACCESSIONS.items() for acc in accs]
if all_accessions:
    print(f'Downloading {len(all_accessions)} SRA accessions...')
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        futures = {executor.submit(download_sra, acc,
                                   RAW_DIR / ds, max(1, CPUS // 2)): (ds, acc)
                   for ds, acc in all_accessions}
        for fut in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
            acc, ok, msg = fut.result()
            if not ok:
                print(f'  FAILED {acc}: {msg}')
    print('Download complete.')
else:
    print('No SRA accessions configured. Add to SRA_ACCESSIONS dict above.')
    print('Alternatively, place pre-downloaded FASTQs directly in RAW_DIR subfolders.')


---
## 3 · QC & Trimming (fastp)

In [ ]:
# fastp: adapter auto-detection, quality trimming, duplicate removal
# Produces per-sample JSON QC reports for downstream MultiQC aggregation

import glob

def run_fastp(r1, r2=None, out_dir=TRIMMED_DIR, qc_dir=QC_DIR, cpus=4):
    sample = Path(r1).stem.replace('_1','').replace('_R1','')
    out_dir.mkdir(parents=True, exist_ok=True)
    qc_dir.mkdir(parents=True,  exist_ok=True)
    out_r1 = out_dir / f'{sample}_trimmed_R1.fastq.gz'
    if out_r1.exists():
        return sample, True, 'skipped'
    cmd = ['fastp',
           '-i', str(r1),
           '-o', str(out_r1),
           '--json', str(qc_dir / f'{sample}_fastp.json'),
           '--html', str(qc_dir / f'{sample}_fastp.html'),
           '--thread', str(cpus),
           '--detect_adapter_for_pe',
           '--correction',
           '--overrepresentation_analysis']
    if r2:
        out_r2 = out_dir / f'{sample}_trimmed_R2.fastq.gz'
        cmd += ['-I', str(r2), '-O', str(out_r2), '--dedup']
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
    return sample, result.returncode == 0, result.stderr[:300]

raw_r1_files = sorted(RAW_DIR.rglob('*_1.fastq*')) + sorted(RAW_DIR.rglob('*_R1*.fastq*'))
print(f'Found {len(raw_r1_files)} R1 FASTQ files for QC/trimming')

trim_results = []
for r1 in raw_r1_files:
    r2_candidates = [r1.parent / r1.name.replace('_1.','_2.').replace('_R1','_R2')]
    r2 = r2_candidates[0] if r2_candidates[0].exists() else None
    sample, ok, msg = run_fastp(r1, r2, cpus=max(1, CPUS // 2))
    trim_results.append({'sample': sample, 'success': ok, 'msg': msg})
    print(f'  {"OK" if ok else "FAIL"}  {sample}')

import pandas as pd
pd.DataFrame(trim_results).to_csv(QC_DIR / 'fastp_summary.csv', index=False)

# Aggregate with MultiQC
subprocess.run(['multiqc', str(QC_DIR), '-o', str(QC_DIR / 'multiqc')],
               capture_output=True)
print('MultiQC report generated.')


---
## 4 · Taxonomic Profiling (MetaPhlAn 4)

In [ ]:
# MetaPhlAn 4: produces species-level relative abundance profiles
# Output: per-sample abundance table → merged into {dataset} species.tsv

METAPHLAN_DB = Path.home() / 'metaphlan_databases'   # downloaded automatically on first run

def run_metaphlan(fastq_r1, fastq_r2=None, out_dir=PROFILE_DIR, cpus=4):
    sample = Path(fastq_r1).stem.replace('_trimmed_R1','')
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / f'{sample}_metaphlan.tsv'
    if out_file.exists():
        return sample, True, 'skipped'
    cmd = ['metaphlan',
           str(fastq_r1),
           '--input_type', 'fastq',
           '--nproc', str(cpus),
           '--bowtie2db', str(METAPHLAN_DB),
           '-o', str(out_file),
           '--unclassified_estimation']
    if fastq_r2:
        cmd = ['metaphlan',
               f'{fastq_r1},{fastq_r2}',
               '--input_type', 'fastq',
               '--nproc', str(cpus),
               '--bowtie2db', str(METAPHLAN_DB),
               '-o', str(out_file),
               '--unclassified_estimation']
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=7200)
    return sample, result.returncode == 0, result.stderr[:200]

trimmed_r1 = sorted(TRIMMED_DIR.rglob('*_trimmed_R1*'))
print(f'Running MetaPhlAn4 on {len(trimmed_r1)} samples...')
for r1 in trimmed_r1:
    r2_cand = r1.parent / r1.name.replace('_R1','_R2')
    r2 = r2_cand if r2_cand.exists() else None
    sample, ok, msg = run_metaphlan(r1, r2, cpus=CPUS)
    print(f'  {"OK" if ok else "FAIL"}  {sample}')

# Merge all profiles into a single species table
indiv_profiles = list(PROFILE_DIR.glob('*_metaphlan.tsv'))
if indiv_profiles:
    result = subprocess.run(
        ['merge_metaphlan_tables.py'] + [str(p) for p in indiv_profiles],
        capture_output=True, text=True)
    if result.returncode == 0:
        merged_path = PROFILE_DIR / 'merged_species.tsv'
        with open(merged_path, 'w') as f:
            f.write(result.stdout)
        print(f'Merged species table: {merged_path}')
        print('Copy merged_species.tsv to Data/ as: {dataset} species.tsv')
    else:
        print(f'Merge failed: {result.stderr[:300]}')


---
## 5 · Functional Profiling (HUMAnN 3)

In [ ]:
# HUMAnN 3: gene family + pathway abundance from MetaPhlAn-aligned reads
# Outputs: {sample}_pathabundance.tsv, {sample}_genefamilies.tsv

def run_humann(fastq_r1, metaphlan_profile, out_dir=PROFILE_DIR / 'humann', cpus=4):
    sample = Path(fastq_r1).stem.replace('_trimmed_R1','')
    out_dir.mkdir(parents=True, exist_ok=True)
    pathway_file = out_dir / f'{sample}_pathabundance.tsv'
    if pathway_file.exists():
        return sample, True, 'skipped'
    cmd = ['humann',
           '--input', str(fastq_r1),
           '--output', str(out_dir),
           '--metaphlan-options', f'--input_type fastq',
           '--taxonomic-profile', str(metaphlan_profile),
           '--threads', str(cpus),
           '--output-basename', sample]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=14400)
    return sample, result.returncode == 0, result.stderr[:300]

humann_out = PROFILE_DIR / 'humann'
humann_out.mkdir(exist_ok=True)
for r1 in trimmed_r1:
    sample = r1.stem.replace('_trimmed_R1','')
    mp_profile = PROFILE_DIR / f'{sample}_metaphlan.tsv'
    if not mp_profile.exists():
        print(f'  SKIP {sample} — MetaPhlAn profile not found')
        continue
    s, ok, msg = run_humann(r1, mp_profile, humann_out, cpus=CPUS)
    print(f'  {"OK" if ok else "FAIL"}  {s}')

# Join pathway tables across samples
pathway_files = list(humann_out.glob('*_pathabundance.tsv'))
if pathway_files:
    merge_cmd = (['humann_join_tables', '--input', str(humann_out),
                  '--output', str(PROFILE_DIR / 'merged_pathabundance.tsv'),
                  '--file_name', 'pathabundance'])
    subprocess.run(merge_cmd, capture_output=True)
    print(f'Merged pathway table: {PROFILE_DIR / "merged_pathabundance.tsv"}')
    print('Copy to Data/ as: {dataset} pathways.tsv  (optional NB01 input)')


---
## 6 · mzML Metabolomics Import & Feature Extraction

In [ ]:
# Parse mzML files using pyteomics/pymzml
# Performs: peak picking, retention time alignment, feature intensity matrix
# Output: metabolite abundance table matching {dataset} mtb.tsv format

try:
    from pyteomics import mzml
    PYTEOMICS = True
except ImportError:
    try:
        import pymzml
        PYTEOMICS = False
    except ImportError:
        print('Install pyteomics or pymzml: pip install pyteomics pymzml')
        PYTEOMICS = None

import numpy as np
import pandas as pd

def extract_features_pyteomics(mzml_path, mz_tol=0.01, rt_tol=30):
    sample = Path(mzml_path).stem
    peaks = []
    with mzml.MzML(str(mzml_path)) as reader:
        for spectrum in reader:
            if spectrum.get('ms level', 0) != 1:
                continue
            rt  = spectrum.get('scanList',{}).get('scan',[{}])[0].get(
                  'scan start time', 0)
            mz_arr  = spectrum.get('m/z array', np.array([]))
            int_arr = spectrum.get('intensity array', np.array([]))
            if len(mz_arr) == 0:
                continue
            top_idx = np.argsort(int_arr)[::-1][:50]  # top 50 peaks per scan
            for idx in top_idx:
                peaks.append({'sample': sample, 'mz': float(mz_arr[idx]),
                              'rt': float(rt), 'intensity': float(int_arr[idx])})
    return pd.DataFrame(peaks)

def extract_features_pymzml(mzml_path):
    sample = Path(mzml_path).stem
    peaks = []
    run = pymzml.run.Reader(str(mzml_path))
    for spectrum in run:
        if spectrum.ms_level != 1:
            continue
        rt = spectrum.scan_time_in_minutes() * 60
        for mz, intensity in zip(spectrum.mz, spectrum.i):
            if intensity > 1000:  # intensity threshold
                peaks.append({'sample': sample, 'mz': float(mz),
                              'rt': float(rt), 'intensity': float(intensity)})
    return pd.DataFrame(peaks)

mzml_files = list(MZML_DIR.glob('*.mzML')) + list(MZML_DIR.glob('*.mzml'))
print(f'Found {len(mzml_files)} mzML files')

all_peaks = []
for mf in mzml_files:
    print(f'  Parsing {mf.name}...')
    if PYTEOMICS is True:
        df = extract_features_pyteomics(mf)
    elif PYTEOMICS is False:
        df = extract_features_pymzml(mf)
    else:
        print('  Skipped — no parser available'); continue
    all_peaks.append(df)
    print(f'    -> {len(df):,} peaks extracted')

if all_peaks:
    peaks_df = pd.concat(all_peaks, ignore_index=True)
    peaks_df.to_parquet(MTB_OUT_DIR / 'raw_peaks.parquet')
    print(f'Total peaks: {len(peaks_df):,} across {peaks_df["sample"].nunique()} samples')
else:
    print('No mzML files processed.')


---
## 7 · mzML Feature Alignment & KEGG Annotation

In [ ]:
# Group peaks into features by m/z + RT proximity
# Annotate features against HMDB / KEGG via m/z matching
# Output: samples x KEGG_ID metabolite abundance matrix

import numpy as np, pandas as pd
from scipy.spatial.distance import cdist

MZ_TOL  = 0.01   # Da  (Orbitrap: 0.001; QToF: 0.01; triple-quad: 0.02)
RT_TOL  = 30     # seconds

def align_features(peaks_df, mz_tol=MZ_TOL, rt_tol=RT_TOL):
    # Simple greedy binning by m/z then RT
    peaks_s = peaks_df.sort_values(['mz','rt']).reset_index(drop=True)
    peaks_s['feature_id'] = -1
    fid = 0
    for i, row in peaks_s.iterrows():
        if peaks_s.at[i,'feature_id'] >= 0:
            continue
        mz_mask = ((peaks_s['mz'] - row['mz']).abs() <= mz_tol)
        rt_mask = ((peaks_s['rt'] - row['rt']).abs() <= rt_tol)
        group   = mz_mask & rt_mask & (peaks_s['feature_id'] < 0)
        peaks_s.loc[group, 'feature_id'] = fid
        fid += 1
    return peaks_s

def annotate_features_kegg(features_df, mz_tol=MZ_TOL):
    # Query KEGG compound database by m/z (requires internet)
    import urllib.request
    unique_mz = features_df.groupby('feature_id')['mz'].mean()
    annotations = {}
    for fid, mz_val in unique_mz.items():
        url = (f'https://rest.kegg.jp/find/compound/{mz_val:.4f}/exact_mass')
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                lines = r.read().decode().strip().split('\n')
            kegg_ids = [l.split('\t')[0].replace('cpd:','') for l in lines if l]
            annotations[fid] = kegg_ids[0] if kegg_ids else f'mz_{mz_val:.4f}'
        except Exception:
            annotations[fid] = f'mz_{mz_val:.4f}'
    return annotations

if 'peaks_df' in locals() and not peaks_df.empty:
    print('Aligning features across samples...')
    peaks_aligned = align_features(peaks_df)
    print(f'  {peaks_aligned["feature_id"].nunique()} unique features identified')

    # Pivot to samples x features intensity matrix
    mtb_matrix = (peaks_aligned
                  .groupby(['sample','feature_id'])['intensity'].max()
                  .unstack(fill_value=0))

    print('Annotating features against KEGG by exact mass...')
    annotations = annotate_features_kegg(peaks_aligned)
    mtb_matrix.columns = [annotations.get(c, f'mz_{c}') for c in mtb_matrix.columns]

    # Save in NB01-compatible TSV format
    dataset_name = 'MY_COHORT_2025'  # update to match your dataset name
    out_mtb  = DATA_DIR / f'{dataset_name} mtb.tsv'
    mtb_matrix.to_csv(out_mtb, sep='\t')
    print(f'Saved metabolite table: {out_mtb}')
    print(f'Shape: {mtb_matrix.shape} (samples x features)')
    print('NB00 complete. Run NB01 to incorporate into the pipeline.')
else:
    print('No peaks data available. Run Cell 6 (mzML parsing) first.')


---
## Appendix · Manual Metadata Template

In [ ]:
# Template for manually creating metadata TSV when SRA metadata is incomplete
import pandas as pd

template_cols = [
    'sample_id',       # must match FASTQ/mzML filename stem
    'study_group',     # e.g. Healthy, CRC, Adenoma
    'Age',             # years (numeric)
    'BMI',             # kg/m2 (numeric)
    'Sex',             # M / F
    'Smoking',         # Yes / No / Former
    'Alcohol',         # Yes / No
    'Tumour_location', # Left / Right / Rectum (if applicable)
    'Stage',           # TNM stage if applicable
    'Country',         # cohort origin
    'Sequencing_platform',  # Illumina HiSeq / NovaSeq etc.
]

template = pd.DataFrame(columns=template_cols)
dataset_name = 'MY_COHORT_2025'
template.to_csv(DATA_DIR / f'{dataset_name} metadata.tsv', sep='\t', index=False)
print(f'Metadata template saved to: {DATA_DIR}/{dataset_name} metadata.tsv')
print('Fill in sample rows then re-run NB01.')
